# FormulaLM — PROMPT 9B: Apples-to-Apples Evaluation of QLoRA v1 vs v2

Based on `notebooks/02_eval_ft_v1.ipynb`. Loads both public Hugging Face adapters pinned to their audited commits on the **exact same `Qwen/Qwen3-1.7B@70d244c` fp16 base** with identical decoding settings (`enable_thinking=False`, `temperature=0`, `max_tokens=96`, `seed=42`, `few_shot=0`, `batch_size=8`):

1. **QLoRA v2 (`abubakar56/formulalm-qwen3-1.7b-lora-v2@e2dd7e81`)** on:
   - `results/ft_v2_val` (`data/processed/val.jsonl`, 500)
   - `results/ft_v2_real` (`data/test_real/test_real_DRAFT.csv`, 80 — _informed by v1 error analysis_)
   - `results/ft_v2_synthetic` (`data/processed/test_synthetic.jsonl`, 500 — _informed by v1 error analysis_)
   - `results/ft_v2_real_v2` (`data/test_real_v2/test_real_v2.csv`, fresh blind set, if present)
2. **Frozen QLoRA v1 (`abubakar56/formulalm-qwen3-1.7b-lora@d9909249`)** ONLY on splits whose results do not exist yet:
   - `results/ft_v1_val` (`data/processed/val.jsonl`, 500)
   - `results/ft_v1_real_v2` (`data/test_real_v2/test_real_v2.csv`, if present)
   - Frozen `results/ft_v1_synthetic` and `results/ft_v1_real` (`v1-frozen`) are never overwritten.
3. **Comparison report (`results/ft_v2_comparison.md`)** with 95% Wilson CIs, per-category tables, and the shipping decision rule.

**Hardware:** Colab or Kaggle T4 GPU (~6–10 minutes total).

In [1]:
# 1. Verify CUDA GPU (must be NVIDIA T4 / CUDA GPU, NOT CPU or TPU)
import torch

assert torch.cuda.is_available(), 'No CUDA GPU detected! Switch runtime to T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

GPU: Tesla T4 | VRAM: 15.64 GB
name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14910 MiB


In [2]:
# 2. Clone / pull FormulaLM (Colab or Kaggle), install eval deps, ensure val.jsonl & test_synthetic.jsonl exist
import getpass, os, pathlib, subprocess, sys

IS_KAGGLE = pathlib.Path('/kaggle/working').is_dir()
REPO_DIR = pathlib.Path('/kaggle/working/FormulaLM' if IS_KAGGLE else '/content/FormulaLM')

if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _ks = UserSecretsClient()
        for _key in ('GITHUB_TOKEN', 'HF_TOKEN'):
            try:
                os.environ[_key] = _ks.get_secret(_key)
                print(f'Loaded Kaggle secret: {_key}')
            except Exception:
                pass
    except Exception as exc:
        print('kaggle_secrets unavailable:', exc)

if not REPO_DIR.exists():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass.getpass('GitHub PAT for abubakarp789/FormulaLM (hidden, empty if public): ').strip()
    url = f'https://{token}@github.com/abubakarp789/FormulaLM.git' if token else 'https://github.com/abubakarp789/FormulaLM.git'
    subprocess.run(['git', 'clone', url, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd {REPO_DIR}
for _p in (str(REPO_DIR), str(REPO_DIR / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
os.environ['PYTHONPATH'] = f"{REPO_DIR}:{REPO_DIR / 'src'}"

%pip uninstall -y -q torchao
%pip install -q formulas openpyxl pandas pyyaml requests tqdm accelerate peft "transformers>=4.51.0" "huggingface_hub>=0.30.0"
%pip install -q --no-deps -e .

hf_token = os.environ.get('HF_TOKEN')
if not hf_token and not IS_KAGGLE:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = None
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)

if not pathlib.Path('data/processed/val.jsonl').is_file() or not pathlib.Path('data/processed/test_synthetic.jsonl').is_file():
    !python -m src.data.build_dataset --version v2 --workers 2

/content/FormulaLM
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/FormulaLM/src/data/build_dataset.py", line 35, in <module>
    from formulalm.config import load_config
ModuleNotFoundError: No module named 'formulalm'


In [ ]:
# 3. Shared identical settings + evaluate QLoRA v2 on val, test_real, test_synthetic, and test_real_v2
import gc, pathlib, torch
from src.eval.run_eval import TransformersBackend, evaluate

MODEL_ID = 'Qwen/Qwen3-1.7B'
BASE_REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
V1_ADAPTER = 'abubakar56/formulalm-qwen3-1.7b-lora'
V1_ADAPTER_REV = 'd9909249c496134ecc72e1d77bdc22e6e0461537'
V2_ADAPTER = 'abubakar56/formulalm-qwen3-1.7b-lora-v2'
V2_ADAPTER_REV = 'e2dd7e8115406ffcdbbdfa60d6f2f10b4aaf2595'

# Identical decoding and precision settings for BOTH v1 and v2:
BACKEND_KWARGS = dict(revision=BASE_REVISION, load_in_4bit=False, enable_thinking=False)
EVAL_KWARGS = dict(seed=42, max_tokens=96, few_shot=0, batch_size=8, resume=True)

BLIND_CSV = pathlib.Path('data/test_real_v2/test_real_v2.csv')

v2_jobs = [
    ('ft_v2_val', 'data/processed/val.jsonl'),
    ('ft_v2_real', 'data/test_real/test_real_DRAFT.csv'),
    ('ft_v2_synthetic', 'data/processed/test_synthetic.jsonl'),
]
if BLIND_CSV.is_file():
    v2_jobs.append(('ft_v2_real_v2', str(BLIND_CSV)))
else:
    print('Note: data/test_real_v2/test_real_v2.csv not present yet; skipping ft_v2_real_v2.')

print(f'Loading v2 adapter: {V2_ADAPTER}@{V2_ADAPTER_REV[:8]} on {MODEL_ID}@{BASE_REVISION[:8]}', flush=True)
v2_backend = TransformersBackend(
    MODEL_ID, adapter_path=V2_ADAPTER, adapter_revision=V2_ADAPTER_REV, **BACKEND_KWARGS
)

for run_name, dataset in v2_jobs:
    print(f'=== Running {run_name} ({dataset}) ===', flush=True)
    m = evaluate(backend=v2_backend, dataset_path=dataset, run_name=run_name, **EVAL_KWARGS)
    ov = m['overall']
    print(
        f"{run_name}: n={ov['n']} | syntax={100*ov['syntax_valid_rate']:.1f}% | "
        f"exec_acc={100*ov['execution_accuracy']:.1f}% | exact={100*ov['exact_match_rate']:.1f}% | "
        f"raw_fmt={100*ov['raw_format_ok_rate']:.1f}% | avg_lat={ov['avg_latency_ms']:.1f}ms",
        flush=True,
    )

del v2_backend
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 4. Apples-to-apples: evaluate frozen v1 on val and test_real_v2 ONLY if those results do not exist yet
# (Frozen results/ft_v1_synthetic and results/ft_v1_real are never modified.)
v1_candidates = [('ft_v1_val', 'data/processed/val.jsonl')]
if BLIND_CSV.is_file():
    v1_candidates.append(('ft_v1_real_v2', str(BLIND_CSV)))

v1_jobs = [
    (run_name, ds)
    for run_name, ds in v1_candidates
    if not (pathlib.Path('results') / run_name / 'metrics.json').is_file()
]

if v1_jobs:
    print(f'Loading v1 adapter: {V1_ADAPTER}@{V1_ADAPTER_REV[:8]} on {MODEL_ID}@{BASE_REVISION[:8]}', flush=True)
    v1_backend = TransformersBackend(
        MODEL_ID, adapter_path=V1_ADAPTER, adapter_revision=V1_ADAPTER_REV, **BACKEND_KWARGS
    )
    for run_name, dataset in v1_jobs:
        print(f'=== Running {run_name} ({dataset}) ===', flush=True)
        m = evaluate(backend=v1_backend, dataset_path=dataset, run_name=run_name, **EVAL_KWARGS)
        ov = m['overall']
        print(
            f"{run_name}: n={ov['n']} | syntax={100*ov['syntax_valid_rate']:.1f}% | "
            f"exec_acc={100*ov['execution_accuracy']:.1f}% | exact={100*ov['exact_match_rate']:.1f}% | "
            f"avg_lat={ov['avg_latency_ms']:.1f}ms",
            flush=True,
        )
    del v1_backend
    gc.collect()
    torch.cuda.empty_cache()
else:
    print('All required v1 apples-to-apples results already exist; nothing to re-run.')

In [ ]:
# 5. Produce v1 vs v2 comparison report with 95% Wilson CIs, per-category tables & decision rule
!python -m src.eval.compare_v1_v2 \
  --results-dir results \
  --out results/ft_v2_comparison.md \
  --json-out results/ft_v2_comparison.json

In [ ]:
# 6. Push lightweight v2 & apples-to-apples v1 evaluation results back to GitHub
PUSH_TO_GITHUB = True
if PUSH_TO_GITHUB:
    !git config user.name "abubakarp789"
    !git config user.email "abubakarp789@gmail.com"
    !git add results/ft_v2_* results/ft_v1_val results/ft_v1_real_v2 2>/dev/null || git add results/ft_v2_*
    !git diff --cached --quiet || git commit -m "results: add QLoRA v2 evaluation and v1-vs-v2 comparison"
    !git pull --rebase origin main
    !git push origin main